In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import shutil

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'

# Importing the PitStrategyNet model
from src.models.model import PitStrategyNet

# Load data
X_train = np.load(processed_dir / 'X_train.npy')
y_train = np.load(processed_dir / 'y_train.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_train: (13362, 15), y_train: (13362, 1)


In [2]:
import yaml
from pathlib import Path

project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

model = PitStrategyNet(
    input_dim    = config['model']['input_dim'],
    hidden_dim_1 = config['model']['hidden_dim_1'],
    hidden_dim_2 = config['model']['hidden_dim_2'],
    dropout_rate = config['model']['dropout_rate']
)

optimizer  = torch.optim.Adam(model.parameters(), lr=config['training']['learning_rate'])
num_epochs = config['training']['epochs']
batch_size = config['training']['batch_size']
criterion = nn.BCEWithLogitsLoss()

In [3]:
class LapDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(LapDataset(X_train, y_train), batch_size=batch_size, shuffle=False)

In [4]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.01

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping to prevent loss going to infinity
        optimizer.step()
        train_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f}")

# Save weights
models_dir = project_root / 'src' / 'models' / config['paths']['folder_name']
models_dir.mkdir(exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': config['model']['input_dim'],
}, models_dir / config['paths']['model_filename'])

shutil.copy(project_root / 'src' / 'config.yaml',
            models_dir / config['paths']['config_name'])

print("Model saved.")

Epoch 10/10 | Train Loss: nan
Model saved.
